# Classical Models

**Navigation**: [← Previous: Images & Labels](01_images.ipynb) | [Next: Deep Learning →](03_deep_learning.ipynb)

A majority dummy, logistic regression on PCA-reduced pixels, and a random forest on HOG / LBP / intensity features.


## Why start simple?

With **546** training images, a model that sees 16,384 raw pixels is over-parameterised. The first two models force a strong inductive bias: (1) a linear decision in a 20–80 dimensional PCA subspace, and (2) engineered edge/texture descriptors that computer vision used for a decade before conv nets. Both train in seconds and give a yardstick the CNNs have to beat.

All reported numbers in this chapter are **test-set** scores. PCA rank and the forest's decision threshold are chosen on the **validation** split only.

In [ ]:

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

PROJ_DIR = Path(".").resolve()
if not (PROJ_DIR / "cancer_cv_utils.py").exists():
    PROJ_DIR = Path("projects/cancer-imaging").resolve()
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

from cancer_cv_utils import (
    CLASS_NAMES,
    METRIC_ORDER,
    artifacts_dir,
    classification_metrics,
    display_plotly,
    extract_cv_features,
    figures_dir,
    grouped_importance,
    load_metrics,
    load_predictions,
    load_splits,
    metrics_frame,
    overlay_heatmap,
    tune_threshold,
)

SPLITS = load_splits()
FIG = figures_dir()
ART = artifacts_dir()
print("Splits:", {k: v["labels"].shape[0] for k, v in SPLITS.items()})
print("Malignant rates:", {k: f"{v['labels'].mean():.1%}" for k, v in SPLITS.items()})


## Dummy classifier

Always predict the majority class (benign / normal). Accuracy looks respectable; cancer detection is zero.

In [ ]:
from sklearn.dummy import DummyClassifier
from IPython.display import Image, display

y_tr, y_te = SPLITS['train']['labels'], SPLITS['test']['labels']
x_tr = SPLITS['train']['images'].reshape(len(y_tr), -1)
x_te = SPLITS['test']['images'].reshape(len(y_te), -1)
dummy = DummyClassifier(strategy='most_frequent', random_state=42).fit(x_tr, y_tr)
dummy_prob = dummy.predict_proba(x_te)[:, 1]
dummy_mets = classification_metrics(y_te, dummy_prob)
pd.Series(dummy_mets)[METRIC_ORDER].to_frame('Dummy (majority)')

## Logistic regression on PCA-reduced pixels

Each image is flattened, standardised, and projected to $k$ principal components. $k$ is picked by validation ROC-AUC from $\{20,40,60,80\}$. Class-weighted logistic regression then predicts malignancy. The decision threshold is tuned on validation F1 so we are not locked to 0.5 on a 27% positive class.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

x_va = SPLITS['val']['images'].reshape(len(SPLITS['val']['labels']), -1)
y_va = SPLITS['val']['labels']
best_auc, best_k = -1, 40
for k in (20, 40, 60, 80):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=k, random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ]).fit(x_tr, y_tr)
    auc = classification_metrics(y_va, pipe.predict_proba(x_va)[:, 1])['roc_auc']
    if auc > best_auc:
        best_auc, best_k = auc, k

logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=best_k, random_state=42)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
]).fit(x_tr, y_tr)
log_val = logreg.predict_proba(x_va)[:, 1]
log_prob = logreg.predict_proba(x_te)[:, 1]
log_t = tune_threshold(y_va, log_val, metric='f1')
log_mets = classification_metrics(y_te, log_prob, threshold=log_t)
print(f'PCA components = {best_k}, val AUC = {best_auc:.3f}, threshold = {log_t:.2f}')
pd.Series(log_mets)[METRIC_ORDER].to_frame('Logistic + PCA')

## HOG + LBP + intensity random forest

**HOG** (histogram of oriented gradients) summarises edge direction in 16×16 cells — useful for mass margins and shadowing. **Uniform LBP** is a local texture histogram. A handful of intensity statistics (mean, centre vs edge, dark-pixel fraction) capture hypoechogenicity. A class-weighted random forest of 200 trees is trained on the concatenated vector.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

x_tr_f, names = extract_cv_features(SPLITS['train']['images'])
x_va_f, _ = extract_cv_features(SPLITS['val']['images'])
x_te_f, _ = extract_cv_features(SPLITS['test']['images'])
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=3,
    class_weight='balanced', random_state=42, n_jobs=-1,
).fit(x_tr_f, y_tr)
rf_val = rf.predict_proba(x_va_f)[:, 1]
rf_prob = rf.predict_proba(x_te_f)[:, 1]
rf_t = tune_threshold(y_va, rf_val, metric='f1')
rf_mets = classification_metrics(y_te, rf_prob, threshold=rf_t)
print(f'{len(names)} features, threshold = {rf_t:.2f}')
pd.Series(rf_mets)[METRIC_ORDER].to_frame('HOG/LBP + RF')

## Side-by-side (this notebook's live fit)

In [ ]:
live = {
    'Dummy (majority)': dummy_mets,
    'Logistic + PCA': log_mets,
    'HOG/LBP + RF': rf_mets,
}
metrics_frame(live).round(3)

Confusion matrices from the committed training run (same seeds, validation-tuned thresholds):

In [ ]:
for fname in ['02_cm_dummy.png', '02_cm_logreg.png', '02_cm_rf.png']:
    display(Image(str(FIG / fname)))

## Takeaways

- The dummy's accuracy is a trap: **recall is 0**.
- PCA-logistic already ranks reasonably (ROC-AUC around 0.80) but is conservative once the F1-tuned threshold is applied — precision over recall.
- Hand-crafted edges lift both ranking quality and F1. That is the baseline a CNN has to beat.

---

**Navigation**: [← Previous: Images & Labels](01_images.ipynb) | [Next: Deep Learning →](03_deep_learning.ipynb)
